In [0]:
%sql
-----  Creating new catalog, schema  ------ 
create catalog if not exists sql_youtube_practise;
use catalog sql_youtube_practise;
create schema if not exists sql;
use sql;
show current schema;

catalog,namespace
sql_youtube_practise,sql


##### Question1: Business city table has data from the day udaan has started operation. 
Write a SQL to identify year-wise count of new cities where udaan started their operations.

In [0]:
%sql
DROP TABLE IF EXISTS business_city;
CREATE TABLE business_city (
    business_date DATE,
    city_id INT
);
INSERT INTO business_city
VALUES
('2020-01-02', 3),
('2020-07-01', 7),
('2021-01-01', 3),
('2021-02-03', 19),
('2022-12-01', 3),
('2022-12-15', 3),
('2022-02-28', 12);

SELECT * FROM business_city ORDER BY business_date;

business_date,city_id
2020-01-02,3
2020-07-01,7
2021-01-01,3
2021-02-03,19
2022-02-28,12
2022-12-01,3
2022-12-15,3


In [0]:
%sql
-- with cte as (
-- select 
--     *,
--     year(business_date) as year,
--     dense_rank() over(partition by city_id order by business_date) as rank
-- from business_city
-- )
-- select 
--     year,
--     count(city_id)
-- from cte 
-- where rank = 1
-- group by year;

with cte as (
    select
        distinct city_id,
        min(business_date) as date
    from business_city
    group by city_id
)
select
    year(date),
    count(city_id) as count
from cte
group by year(date);

year(date),count
2020,2
2021,1
2022,1


##### Question2: There are 3 rows in a movie hall each with 10 seats in each row. write a sql query to find 4 consecutive empty seats.

In [0]:
%sql
DROP TABLE IF EXISTS movie;
CREATE TABLE movie (seat VARCHAR(50), occupancy INT);
INSERT INTO movie VALUES
('a1',1), ('a2',1), ('a3',0), ('a4',0), ('a5',0), ('a6',0), ('a7',1), ('a8',1), ('a9',0), ('a10',0), ('b1',0), ('b2',0), ('b3',0), ('b4',1), ('b5',1), ('b6',1), ('b7',1), ('b8',0), ('b9',0), ('b10',0), ('c1',0), ('c2',1), ('c3',0), ('c4',1), ('c5',1), ('c6',0), ('c7',1), ('c8',0), ('c9',0), ('c10',1);

SELECT * FROM movie;

seat,occupancy
a1,1
a2,1
a3,0
a4,0
a5,0
a6,0
a7,1
a8,1
a9,0
a10,0


In [0]:
%sql
with cte as (
    select
        seat,
        left(seat,1) as row,
        substr(seat,2) as seat_no,
        occupancy
    from movie
)
select *,
    case
        when (seat_no - occupancy) = seat_no then "empty"
        else "occupied"
    end as status
from cte;

seat,row,seat_no,occupancy,status
a1,a,1,1,occupied
a2,a,2,1,occupied
a3,a,3,0,empty
a4,a,4,0,empty
a5,a,5,0,empty
a6,a,6,0,empty
a7,a,7,1,occupied
a8,a,8,1,occupied
a9,a,9,0,empty
a10,a,10,0,empty


##### Question 4: determine phone numbers that satisfy below conditions: 
> - the numbers have both incoming & outgoing calls
> - the sum of duration of outgoing calls should be greater than sum of duration of incoming calls

In [0]:
%sql
DROP TABLE IF EXISTS call_details;
CREATE TABLE call_details (
    call_type VARCHAR(10),
    call_number VARCHAR(12),
    call_duration INT
);
INSERT INTO call_details VALUES ('OUT','181868',13), ('OUT','2159010',8), ('OUT','2159010',178), ('SMS','4153810',1), ('OUT','2159010',152), ('OUT','9140152',18), ('SMS','4162672',1), ('SMS','9168204',1), ('OUT','9168204',576), ('INC','2159010',5), ('INC','2159010',4), ('SMS','2159010',1), ('SMS','4535614',1), ('OUT','181868',20), ('INC','181868',54), ('INC','218748',20), ('INC','2159010',9), ('INC','197432',66), ('SMS','2159010',1), ('SMS','4535614',1);

SELECT * FROM call_details;

call_type,call_number,call_duration
OUT,181868,13
OUT,2159010,8
OUT,2159010,178
SMS,4153810,1
OUT,2159010,152
OUT,9140152,18
SMS,4162672,1
SMS,9168204,1
OUT,9168204,576
INC,2159010,5


In [0]:
%sql
with cte as (
    select
        call_number,
        sum(case when call_type = 'OUT' then call_duration else null end) as out_duration,
        sum(case when call_type = 'INC' then call_duration else null end) as inc_duration
    from call_details
    group by call_number
)
select * from cte
where out_duration is not null and inc_duration is not null and out_duration > inc_duration;

call_number,out_duration,inc_duration
2159010,338,18
